In [ ]:
import os
import glob
import pickle
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pandas as pd
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql.functions import col
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.calibration import calibration_curve
import model_inference


## Setup spark session 

In [6]:
spark = pyspark.sql.SparkSession.builder \
.appName("dev") \
.master("local[*]") \
.getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

## setting up the config

In [8]:
model_name = "credit_model_2024_09_01.pkl"
snapshote_date_str = "2024-01-01"

In [9]:
config = {}
config["snapshot_date_str"] = snapshote_date_str
config["snapshot_date"] = datetime.strptime(config["snapshot_date_str"], "%Y-%m-%d")
config["model_name"] = model_name
config["model_bank_directory"] = "model_bank/"
config["model_artefact_filepath"] = config["model_bank_directory"] + config["model_name"]


## Load the model

In [10]:
with open(config["model_artefact_filepath"], 'rb') as file:
    model_artefact = pickle.load(file)

print("Model loaded successfully!" + config["model_artefact_filepath"])

Model loaded successfully!model_bank/credit_model_2024_09_01.pkl


## load feature store

In [15]:
feature_location = "datamart/gold/features_store/"

feature_store_sdf = spark.read.option("header", "true").parquet(feature_location +"gold_features_store_2024_01_01.parquet")
feature_store_sdf.show()

+-----------+-------------+---+-------------+-------------+---------------------+-----------------+---------------+-------------+-----------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+---------------------+-------------------+-----------------------+---------------+-----------+-------------+---------------------+---------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------+
|customer_id|snapshot_date|age|   occupation|annual_income|monthly_inhand_salary|num_bank_accounts|num_credit_card|interest_rate|num_of_loan|delay_from_due_date|num_of_dela

## Preprocessing step

In [19]:
feature_pdf = feature_store_sdf.toPandas()

feature_cols = [
    col
    for col in feature_pdf.columns
    if col not in ["customer_id", "snapshot_date"]
]

X_inference = feature_pdf[feature_cols]
transformer = model_artefact['preprocessing_transformers']['transformer']
X_inference = transformer.transform(X_inference)
X_inference


array([[-0.36873769, -0.05690128,  0.99423588, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.76341118, -0.09882894, -0.42149799, ...,  0.        ,
         1.        ,  0.        ],
       [-1.29504131, -0.09339124, -0.13069825, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.88335081, -0.11479608, -0.87426674, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.86633381, -0.01546357,  2.5263387 , ...,  0.        ,
         1.        ,  0.        ],
       [-1.7067318 , -0.07876988,  0.33799399, ...,  0.        ,
         0.        ,  1.        ]], shape=(485, 64))

### Model prediciont inference

In [22]:
model = model_artefact["model"]
y_inference = model.predict_proba(X_inference)[:, 1]

y_inference_pdf = feature_pdf[["customer_id", "snapshot_date"]].copy()
y_inference_pdf["model_name"] = config["model_name"]
y_inference_pdf["model_predictions"] = y_inference
y_inference_pdf

,customer_id,snapshot_date,model_name,model_predictions
0,CUS_0x102d,2024-01-01,credit_model_2024_09_01.pkl,0.138283
1,CUS_0x1051,2024-01-01,credit_model_2024_09_01.pkl,0.161370
2,CUS_0x1269,2024-01-01,credit_model_2024_09_01.pkl,0.057678
3,CUS_0x1290,2024-01-01,credit_model_2024_09_01.pkl,0.112802
4,CUS_0x12d1,2024-01-01,credit_model_2024_09_01.pkl,0.158913
...,...,...,...,...
480,CUS_0xc68d,2024-01-01,credit_model_2024_09_01.pkl,0.093662
481,CUS_0xc6d8,2024-01-01,credit_model_2024_09_01.pkl,0.138930
482,CUS_0xe8d,2024-01-01,credit_model_2024_09_01.pkl,0.694807
483,CUS_0xf9e,2024-01-01,credit_model_2024_09_01.pkl,0.157359


## Compare raw vs calibrated model

In [ ]:
compare_models = {
    "raw": "credit_model_2024_09_01.pkl",
    "calibrated": "credit_model_2024_09_01_calibrated.pkl",
}

# score the same feature snapshot with each model, using each artefact's own transformer and columns
comparison_pdf = feature_pdf[["customer_id", "snapshot_date"]].copy()
for label, name in compare_models.items():
    with open(config["model_bank_directory"] + name, 'rb') as file:
        artefact = pickle.load(file)
    X = artefact['preprocessing_transformers']['transformer'].transform(feature_pdf[artefact['feature_cols']])
    comparison_pdf[label] = artefact['model'].predict_proba(X)[:, 1]

# loans starting on this snapshot get their 30dpd_6mob label six months later
label_files = glob.glob("datamart/gold/label_store/*.parquet")
labels_pdf = (
    spark.read.parquet(*label_files)
    .filter(col("feature_snapshot_date") == snapshote_date_str)
    .select("customer_id", "label")
    .toPandas()
)
comparison_pdf = comparison_pdf.merge(labels_pdf, on="customer_id", how="inner")
print("scored loans with a label:", len(comparison_pdf), "/", len(feature_pdf))

comparison_pdf

In [ ]:
y_true = comparison_pdf["label"]

# AUC: ranking (should be close); Brier / log loss: probability quality (lower is better)
metrics_pdf = pd.DataFrame({
    label: {
        "auc": roc_auc_score(y_true, comparison_pdf[label]),
        "brier": brier_score_loss(y_true, comparison_pdf[label]),
        "log_loss": log_loss(y_true, comparison_pdf[label]),
        "mean_prediction": comparison_pdf[label].mean(),
        "actual_default_rate": y_true.mean(),
    }
    for label in compare_models
})
display(metrics_pdf.round(4))

fig, (ax_curve, ax_hist) = plt.subplots(1, 2, figsize=(11, 4.5))
for label in compare_models:
    frac_pos, mean_pred = calibration_curve(y_true, comparison_pdf[label], n_bins=5, strategy='quantile')
    ax_curve.plot(mean_pred, frac_pos, marker='o', label=label)
    ax_hist.hist(comparison_pdf[label], bins=20, alpha=0.5, label=label)

ax_curve.plot([0, 1], [0, 1], '--', color='grey', label='perfect')
ax_curve.set_xlabel('Mean predicted probability')
ax_curve.set_ylabel('Observed default rate')
ax_curve.set_title('Calibration')
ax_curve.legend()
ax_hist.set_xlabel('Predicted default probability')
ax_hist.set_ylabel('Loans')
ax_hist.set_title('Score distribution')
ax_hist.legend()
plt.show()

## backfill

In [24]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

In [25]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
